## Block Cipher Project

Names: Lucie Spanbauer, Ashraf Mirza, Monish Sreekanth, kmm9660

## Problem 1 – Modes of Operation

In [82]:
## Code

## Problem 2 – Mini Block Cipher

The Mini Block Cipher is based on Simplified AES (SAES).

The plaintext and key are both 16 bits.

The original SAES lecture example uses:
- 16-bit plaintext
- 16-bit key
- 4-bit S-box values
- key expansion into W[0] through W[5]
- round keys K0, K1, and K2

For this assignment, the initial AddRoundKey operation is removed.

In [83]:
SBOX = {
    '0000': '1001',
    '0001': '0100',
    '0010': '1010',
    '0011': '1011',
    '0100': '1101',
    '0101': '0001',
    '0110': '1000',
    '0111': '0101',
    '1000': '0110',
    '1001': '0010',
    '1010': '0000',
    '1011': '0011',
    '1100': '1100',
    '1101': '1110',
    '1110': '1111',
    '1111': '0111'
}

INV_SBOX = {v: k for k, v in SBOX.items()}

In [84]:
def sub_nibbles(state):
    return ''.join(SBOX[state[i:i+4]] for i in range(0, 16, 4))


def inv_sub_nibbles(state):
    return ''.join(INV_SBOX[state[i:i+4]] for i in range(0, 16, 4))


def shift_rows(state):
    # State:
    # a b
    # c d
    #
    # Shift bottom row left:
    # a b
    # d c

    return state[0:4] + state[12:16] + state[8:12] + state[4:8]


def inv_shift_rows(state):
    return shift_rows(state)


def xor_bits(a, b):
    return ''.join(str(int(x) ^ int(y)) for x, y in zip(a, b))

In [85]:
def gf_multiply(a, b):
    result = 0

    for _ in range(4):
        if b & 1:
            result ^= a

        high_bit = a & 0b1000
        a <<= 1

        if high_bit:
            a ^= 0b10011

        b >>= 1

    return result & 0b1111


def mix_columns(state):
    a = int(state[0:4], 2)
    b = int(state[4:8], 2)
    c = int(state[8:12], 2)
    d = int(state[12:16], 2)

    r1 = gf_multiply(a, 1) ^ gf_multiply(c, 4)
    r2 = gf_multiply(a, 4) ^ gf_multiply(c, 1)

    r3 = gf_multiply(b, 1) ^ gf_multiply(d, 4)
    r4 = gf_multiply(b, 4) ^ gf_multiply(d, 1)

    return (
        format(r1, '04b') +
        format(r3, '04b') +
        format(r2, '04b') +
        format(r4, '04b')
    )

In [86]:
def inv_mix_columns(state):
    a = int(state[0:4], 2)
    b = int(state[4:8], 2)
    c = int(state[8:12], 2)
    d = int(state[12:16], 2)

    r1 = gf_multiply(a, 9) ^ gf_multiply(c, 2)
    r2 = gf_multiply(a, 2) ^ gf_multiply(c, 9)

    r3 = gf_multiply(b, 9) ^ gf_multiply(d, 2)
    r4 = gf_multiply(b, 2) ^ gf_multiply(d, 9)

    return (
        format(r1, '04b') +
        format(r3, '04b') +
        format(r2, '04b') +
        format(r4, '04b')
    )

## Key Expansion

The 16-bit key is divided into two 8-bit words, W0 and W1.
Key expansion generates W2, W3, W4, and W5.

The round keys are:

K0 = W0 || W1

K1 = W2 || W3

K2 = W4 || W5

In [87]:
RCON1 = '1000'
RCON2 = '0011'


def rot_nib(word):
    return word[4:8] + word[0:4]


def sub_nib(word):
    return SBOX[word[0:4]] + SBOX[word[4:8]]


def key_expansion(key):
    W0 = key[0:8]
    W1 = key[8:16]

    temp = sub_nib(rot_nib(W1))
    W2 = xor_bits(W0, xor_bits(RCON1 + '0000', temp))
    W3 = xor_bits(W1, W2)

    temp = sub_nib(rot_nib(W3))
    W4 = xor_bits(W2, xor_bits(RCON2 + '0000', temp))
    W5 = xor_bits(W3, W4)

    K0 = W0 + W1
    K1 = W2 + W3
    K2 = W4 + W5

    return K0, K1, K2

In [88]:
key = '0101100101111010'

K0, K1, K2 = key_expansion(key)

print("K0 =", K0)
print("K1 =", K1)
print("K2 =", K2)

K0 = 0101100101111010
K1 = 1101110010100110
K2 = 0110110011001010


## Task 1a- Mini Block Cipher Encryption and Decryption

The Mini Block Cipher uses two encryption rounds.

Round 1 using K1:
1. Substitute
2. ShiftRows
3. MixColumns
4. AddRoundKey

Round 2 using K2:
1. Substitute
2. ShiftRows
3. AddRoundKey

The initial AddRoundKey using K0 is omitted.

Decryption performs the inverse operations in reverse order.

In [89]:
def add_round_key(state, key):
    return xor_bits(state, key)


def encrypt_round1(state, K1):
    state = sub_nibbles(state)
    state = shift_rows(state)
    state = mix_columns(state)
    state = add_round_key(state, K1)

    return state


def encrypt_round2(state, K2):
    state = sub_nibbles(state)
    state = shift_rows(state)
    state = add_round_key(state, K2)

    return state


def decrypt_round2(state, K2):
    state = add_round_key(state, K2)
    state = inv_shift_rows(state)
    state = inv_sub_nibbles(state)

    return state


def decrypt_round1(state, K1):
    state = add_round_key(state, K1)
    state = inv_mix_columns(state)
    state = inv_shift_rows(state)
    state = inv_sub_nibbles(state)

    return state

In [90]:
def mini_encrypt(plaintext, key):
    K0, K1, K2 = key_expansion(key)

    # Initial AddRoundKey is intentionally removed
    state = encrypt_round1(plaintext, K1)
    ciphertext = encrypt_round2(state, K2)

    return ciphertext


def mini_decrypt(ciphertext, key):
    K0, K1, K2 = key_expansion(key)

    state = decrypt_round2(ciphertext, K2)
    plaintext = decrypt_round1(state, K1)

    return plaintext

In [91]:
key = '0101100101111010'
plaintext = '0101101001000011'

ciphertext = mini_encrypt(plaintext, key)
recovered = mini_decrypt(ciphertext, key)

print("Plaintext :", plaintext)
print("Ciphertext:", ciphertext)
print("Recovered :", recovered)
print("Correct?  :", plaintext == recovered)

Plaintext : 0101101001000011
Ciphertext: 1000000001111111
Recovered : 0101101001000011
Correct?  : True


## Task 1b- Ten Plaintext/Ciphertext Pairs

The implementation is tested using at least ten different plaintext blocks.

Each plaintext is encrypted using the same 16-bit key. The resulting ciphertexts are recorded as plaintext/ciphertext pairs.

In [92]:
# Fixed 16-bit key
key = '0101100101111010'

# Ten different 16-bit plaintexts
plaintexts = [
    '0000000000000000',
    '0001000100010001',
    '0010001000100010',
    '0011001100110011',
    '0100010001000100',
    '0101010101010101',
    '0110011001100110',
    '0111011101110111',
    '1000100010001000',
    '1001100110011001'
]

# Generate the plaintext-ciphertext pairs
pairs = []

for plaintext in plaintexts:
    ciphertext = mini_encrypt(plaintext, key)
    pairs.append((plaintext, ciphertext))

# Display the pairs
for i, (plaintext, ciphertext) in enumerate(pairs, 1):
    print(f"Pair {i}:")
    print("Plaintext :", plaintext)
    print("Ciphertext:", ciphertext)
    print()

Pair 1:
Plaintext : 0000000000000000
Ciphertext: 1110001010001111

Pair 2:
Plaintext : 0001000100010001
Ciphertext: 0110100000101001

Pair 3:
Plaintext : 0010001000100010
Ciphertext: 0100011000111100

Pair 4:
Plaintext : 0011001100110011
Ciphertext: 1010100111110100

Pair 5:
Plaintext : 0100010001000100
Ciphertext: 0010110001000011

Pair 6:
Plaintext : 0101010101010101
Ciphertext: 0000011110111000

Pair 7:
Plaintext : 0110011001100110
Ciphertext: 1101101000010000

Pair 8:
Plaintext : 0111011101110111
Ciphertext: 0001000110100101

Pair 9:
Plaintext : 1000100010001000
Ciphertext: 1111111110011110

Pair 10:
Plaintext : 1001100110011001
Ciphertext: 0011000001010010



## Verification of Encryption and Decryption

Each ciphertext is decrypted using the same key. The recovered plaintext is compared with the original plaintext to verify that encryption and decryption are inverses.

In [93]:
# Verify that encryption and decryption work for all 10 pairs

for i, (plaintext, ciphertext) in enumerate(pairs, 1):
    recovered = mini_decrypt(ciphertext, key)

    print(f"Pair {i}:")
    print("  Plaintext :", plaintext)
    print("  Ciphertext:", ciphertext)
    print("  Recovered :", recovered)
    print("  Correct   :", plaintext == recovered)
    print()

Pair 1:
  Plaintext : 0000000000000000
  Ciphertext: 1110001010001111
  Recovered : 0000000000000000
  Correct   : True

Pair 2:
  Plaintext : 0001000100010001
  Ciphertext: 0110100000101001
  Recovered : 0001000100010001
  Correct   : True

Pair 3:
  Plaintext : 0010001000100010
  Ciphertext: 0100011000111100
  Recovered : 0010001000100010
  Correct   : True

Pair 4:
  Plaintext : 0011001100110011
  Ciphertext: 1010100111110100
  Recovered : 0011001100110011
  Correct   : True

Pair 5:
  Plaintext : 0100010001000100
  Ciphertext: 0010110001000011
  Recovered : 0100010001000100
  Correct   : True

Pair 6:
  Plaintext : 0101010101010101
  Ciphertext: 0000011110111000
  Recovered : 0101010101010101
  Correct   : True

Pair 7:
  Plaintext : 0110011001100110
  Ciphertext: 1101101000010000
  Recovered : 0110011001100110
  Correct   : True

Pair 8:
  Plaintext : 0111011101110111
  Ciphertext: 0001000110100101
  Recovered : 0111011101110111
  Correct   : True

Pair 9:
  Plaintext : 1000100010

# Task 2- Meet-in-the-Middle Attack

Because the Mini Block Cipher consists of two rounds using independent round keys K1 and K2, a Meet-in-the-Middle attack can be used.

For every possible K1:

X = encrypt_round1(P, K1)

For every possible K2:

X' = decrypt_round2(C, K2)

A matching intermediate value X = X' identifies a candidate pair (K1, K2).

Additional plaintext/ciphertext pairs are then used to eliminate false matches.

In [94]:
def mitm_attack(plaintext, ciphertext):
    forward = {}

    # Try every possible K1
    for k1_int in range(65536):
        K1 = format(k1_int, '016b')
        intermediate = encrypt_round1(plaintext, K1)

        forward.setdefault(intermediate, []).append(K1)

    matches = []

    # Try every possible K2
    for k2_int in range(65536):
        K2 = format(k2_int, '016b')
        intermediate = decrypt_round2(ciphertext, K2)

        if intermediate in forward:
            for K1 in forward[intermediate]:
                matches.append((K1, K2))

    return matches

In [95]:
candidates = mitm_attack(pairs[0][0], pairs[0][1])

print("Candidates after first pair:", len(candidates))

Candidates after first pair: 65536


## Task 2b- Eliminating False Matches

A single plaintext/ciphertext pair may produce multiple candidate key pairs.

Additional plaintext/ciphertext pairs are used to test the candidates.

For each candidate (K1, K2), the plaintext is encrypted using the candidate keys. The candidate is retained only if the resulting ciphertext matches the known ciphertext.

In [96]:
for plaintext, ciphertext in pairs[1:]:
    candidates = [
        (K1, K2)
        for K1, K2 in candidates
        if encrypt_round2(
            encrypt_round1(plaintext, K1),
            K2
        ) == ciphertext
    ]

    print("Remaining candidates:", len(candidates))

Remaining candidates: 16
Remaining candidates: 1
Remaining candidates: 1
Remaining candidates: 1
Remaining candidates: 1
Remaining candidates: 1
Remaining candidates: 1
Remaining candidates: 1
Remaining candidates: 1


## Recovered Key Pair

After testing the candidate key pairs against multiple known plaintext/ciphertext pairs, the remaining candidate should correspond to the actual round keys K1 and K2 used for encryption.

In [97]:
print("Final candidate key pair(s):")

for K1, K2 in candidates:
    print("K1 =", K1)
    print("K2 =", K2)

Final candidate key pair(s):
K1 = 1101110010100110
K2 = 0110110011001010


# Task 3- Complexity Analysis

## 3a- Key Space

The Mini Block Cipher uses a 16-bit key.

Therefore, the total number of possible keys is:

2^16 = 65,536

Thus, the key space contains 65,536 possible keys.

## Task 3b – Meet-in-the-Middle Complexity

The MITM attack searches all possible values of K1 and all possible values of K2.

For each possible K1, one intermediate value is computed.

For each possible K2, one intermediate value is computed.

Therefore:

2^16 + 2^16
= 2^17
= 131,072

The MITM attack requires approximately 131,072 operations, in addition to the memory required to store the intermediate values.

## Task 3c – Exhaustive Search Complexity

An exhaustive search would test every possible combination of K1 and K2.

There are:

2^16 possible K1 values

and

2^16 possible K2 values.

Therefore:

2^16 × 2^16
= 2^32
= 4,294,967,296

Thus, exhaustive search requires approximately 4.29 billion key-pair combinations.

## Task 3d – Tradeoff

The Meet-in-the-Middle attack reduces the computational complexity from approximately 2^32 to approximately 2^17 operations.

However, this improvement requires additional memory to store the intermediate values.

Therefore, the MITM attack trades memory for reduced computation.

# Task 4 – Improving the Mini Block Cipher

The Mini Block Cipher could be strengthened in several ways.

1. Increase the key size.
   A larger key space would make exhaustive key searches more difficult.

2. Increase the number of rounds.
   More rounds would make the cipher more complex and increase the difficulty of cryptanalysis.

3. Avoid a simple meet-in-the-middle structure.
   The cipher should be designed so that an attacker cannot easily divide the encryption into independent halves and match intermediate states.

4. Strengthen confusion and diffusion.
   Stronger nonlinear substitution and better diffusion would make relationships between plaintext, ciphertext, and keys more difficult to analyze.

5. Use a modern standardized block cipher construction.
   Instead of a simplified educational cipher, a well-studied construction such as AES provides substantially stronger security.